In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models # UPGRADE 1: Load ResNet
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import f1_score

# --- CONFIGURATION ---
# CFG = {
#     'batch_size': 64,      
#     'epochs': 50,
#     'lr': 5e-4,            # Giữ nguyên LR cơ bản cho các lớp mới
#     'weight_decay': 1e-4,
#     'image_size': 224,     
#     'num_classes': 7,
#     'mixup_alpha': 0.1,    # Bật mixup nhẹ (tùy chọn, mặc định theo code gốc của bạn)
#     'label_smoothing': 0.05,
#     'focal_gamma': 2.0,    
#     'device': 'cuda' if torch.cuda.is_available() else 'cpu'
# }
# --- CONFIGURATION ---
CFG = {
    'batch_size': 64,      
    'epochs': 50,
    'lr': 5e-4,            
    'weight_decay': 1e-4,
    'image_size': 224,     
    'num_classes': 7,
    'mixup_alpha': 0.0,    # [SỬA] Bắt đầu bằng 0 (Tắt MixUp)
    'erasing_p': 0.1,      # [THÊM] Tham số xóa ảnh bắt đầu cực thấp
    'label_smoothing': 0.05,
    'focal_gamma': 2.0,    
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}
print(f"🚀 Đang sử dụng thiết bị: {CFG['device']}")
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
DATA_DIR = '/kaggle/input/datasets/msambare/fer2013' # Đổi theo path thực tế

# --- TRANSFORMS & DATALOADER ---
# train_transform = transforms.Compose([
#     transforms.Resize((CFG['image_size'], CFG['image_size'])),
#     transforms.Grayscale(num_output_channels=3),                
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(10),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), 
#     transforms.RandomErasing(p=0.2, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0) 
# ])
train_transform = transforms.Compose([
    transforms.Resize((CFG['image_size'], CFG['image_size'])),
    transforms.Grayscale(num_output_channels=3),                
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), 
    # [SỬA] Sử dụng biến để sau này dễ can thiệp đổi tỷ lệ
    transforms.RandomErasing(p=CFG['erasing_p'], scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0) 
])

val_transform = transforms.Compose([
    transforms.Resize((CFG['image_size'], CFG['image_size'])),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f"📦 Dữ liệu Train: {len(train_dataset)} | Validation: {len(val_dataset)}")

In [ ]:
class FocalLoss(nn.Module):
    """[ĐÃ SỬA] Focal Loss tách biệt tính pt và CE loss để tương thích với Label Smoothing"""
    def __init__(self, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma # Sửa typo gammaa
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        # 1. Tính CE Loss không có label smoothing để lấy pt chuẩn
        ce_loss_standard = F.cross_entropy(inputs, targets, reduction='none', label_smoothing=0.0)
        pt = torch.exp(-ce_loss_standard)
        
        # 2. Tính CE Loss CÓ label smoothing làm base loss
        ce_loss_smooth = F.cross_entropy(inputs, targets, reduction='none', label_smoothing=self.label_smoothing)
        
        # 3. Áp dụng Focal Loss
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss_smooth
        return focal_loss.mean()

def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
class ResNetTokenizer(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-2])
        
    def forward(self, x):
        x = self.feature_extractor(x) 
        x = x.flatten(2).transpose(1, 2) 
        return x

class SeqPool(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.attention_pool = nn.Linear(embedding_dim, 1)

    def forward(self, x):
        attn_weights = F.softmax(self.attention_pool(x), dim=1)
        pooled_x = torch.matmul(attn_weights.transpose(1, 2), x).squeeze(1)
        return pooled_x

class CCT_ResNet(nn.Module):
    def __init__(self, num_classes=7, embed_dim=512, num_layers=4, num_heads=8):
        super().__init__()
        self.tokenizer = ResNetTokenizer()
        
        # [ĐÃ SỬA] Khởi tạo Positional Embedding bằng phân phối chuẩn (nhiễu nhỏ)
        self.positional_emb = nn.Parameter(torch.randn(1, 49, embed_dim) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=embed_dim*4,
            dropout=0.1,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.seq_pool = SeqPool(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)
        
    def forward(self, x):
        x = self.tokenizer(x)
        x = x + self.positional_emb 
        x = self.transformer(x)
        x = self.seq_pool(x)
        x = self.classifier(x)
        return x

model = CCT_ResNet(num_classes=CFG['num_classes']).to(CFG['device'])
print(f"🧠 Số lượng tham số của mô hình: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
criterion = FocalLoss(gamma=CFG['focal_gamma'], label_smoothing=CFG['label_smoothing'])

optimizer = optim.AdamW([
    {'params': model.tokenizer.parameters(), 'lr': 1e-5},       # Bảo vệ pre-trained weights
    {'params': model.transformer.parameters(), 'lr': CFG['lr']}, # Các lớp xây mới học với LR chuẩn
    {'params': model.positional_emb, 'lr': CFG['lr']},
    {'params': model.seq_pool.parameters(), 'lr': CFG['lr']},
    {'params': model.classifier.parameters(), 'lr': CFG['lr']}
], weight_decay=CFG['weight_decay'])

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])

# --- TRAINING LOOP ---
# best_acc = 0.0

# for epoch in range(1, CFG['epochs'] + 1):
#     model.train()
#     train_loss = 0.0
#     train_correct = 0
#     train_total = 0
    
#     pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{CFG["epochs"]} [Train]')
#     for inputs, targets in pbar:
#         inputs, targets = inputs.to(CFG['device']), targets.to(CFG['device'])
        
#         inputs_mixed, targets_a, targets_b, lam = mixup_data(inputs, targets, CFG['mixup_alpha'])
        
#         optimizer.zero_grad()
#         outputs = model(inputs_mixed)
        
#         loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
#         loss.backward()
        
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         optimizer.step()
        
#         train_loss += loss.item() * inputs_mixed.size(0)
        
#         _, predicted = outputs.max(1)
#         train_total += targets.size(0)
#         train_correct += predicted.eq(targets).sum().item()
        
#         pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
#     scheduler.step()
# --- TRAINING LOOP ---
best_acc = 0.0
hard_mode_activated = False # [THÊM] Cờ theo dõi trạng thái

for epoch in range(1, CFG['epochs'] + 1):
    
    # ==============================================================
    # 🔥 CURRICULUM LEARNING LOGIC (TỰ ĐỘNG NÂNG ĐỘ KHÓ)
    # ==============================================================
    if best_acc >= 50.0 and not hard_mode_activated:
        print("\n🚀 ĐỘ CHÍNH XÁC VƯỢT 50%! KÍCH HOẠT 'HARD MODE'...")
        # 1. Bật MixUp với cường độ mạnh
        current_mixup_alpha = 0.2 
        # 2. Tăng xác suất Random Erasing lên 50%
        # transform[-1] trỏ đúng vào dòng RandomErasing ở vị trí cuối cùng
        train_dataset.transform.transforms[-1].p = 0.5 
        hard_mode_activated = True
        
    elif not hard_mode_activated:
        # Nếu chưa tới 50%, tiếp tục tắt MixUp
        current_mixup_alpha = 0.0
    # ==============================================================

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{CFG["epochs"]} [Train]')
    for inputs, targets in pbar:
        inputs, targets = inputs.to(CFG['device']), targets.to(CFG['device'])
        
        # [SỬA] Truyền current_mixup_alpha thay vì CFG['mixup_alpha']
        inputs_mixed, targets_a, targets_b, lam = mixup_data(inputs, targets, current_mixup_alpha)
        
        optimizer.zero_grad()
        outputs = model(inputs_mixed)
        
        loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs_mixed.size(0)
        
        # Code tính Train_Acc của bạn giữ nguyên ở đây...
        _, predicted = outputs.max(1)
        train_total += targets.size(0)
        train_correct += predicted.eq(targets).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
    scheduler.step()
    
    #... [Phần Code tính Val_Acc, MacroF1 và Save Model của bạn giữ nguyên] ...
    
    epoch_train_loss = train_loss / len(train_loader.dataset)
    epoch_train_acc = 100. * train_correct / train_total
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    all_targets = []
    all_preds = []
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(CFG['device']), targets.to(CFG['device'])
            outputs = model(inputs)
            
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
            
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()
            
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    
    epoch_val_loss = val_loss / len(val_loader.dataset)
    epoch_val_acc = 100. * val_correct / val_total
    epoch_val_macro_f1 = 100. * f1_score(all_targets, all_preds, average='macro')
    
    print(f'➡️ Epoch {epoch} | Tr_Loss: {epoch_train_loss:.4f} | Val_Loss: {epoch_val_loss:.4f} | Tr_Acc: {epoch_train_acc:.2f}% | Val_Acc: {epoch_val_acc:.2f}% | Val_MacroF1: {epoch_val_macro_f1:.2f}%')
    
    if epoch_val_acc > best_acc:
        best_acc = epoch_val_acc
        torch.save(model.state_dict(), 'cct_resnet_fer2013_best.pth')
        print(f"⭐ Đã lưu model tốt nhất (Val Acc: {best_acc:.2f}%)")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
def evaluate_fer2013(model, dataloader, device):
    """
    Hàm đánh giá mô hình phân loại cảm xúc FER2013 (7 class).
    In ra Classification Report và vẽ Confusion Matrix.
    """
    model.eval()
    all_targets = []
    all_preds = []
    
    print("⏳ Đang tính toán dự đoán trên tập dữ liệu. Vui lòng đợi...")
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            
            # Lưu lại kết quả để tính toán metric
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            
    # Tự động lấy tên 7 cảm xúc từ thuộc tính 'classes' của ImageFolder Dataset
    # Thông thường với FER2013 sẽ là: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
    class_names = dataloader.dataset.classes
    
    # ---------------------------------------------------------
    # 1. IN CLASSIFICATION REPORT
    # ---------------------------------------------------------
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT - FER2013".center(60))
    print("="*60 + "\n")
    
    report = classification_report(
        all_targets, 
        all_preds, 
        target_names=class_names, 
        digits=4 # Lấy 4 chữ số thập phân cho chi tiết
    )
    print(report)
    
    # ---------------------------------------------------------
    # 2. VẼ CONFUSION MATRIX (HEATMAP)
    # ---------------------------------------------------------
    cm = confusion_matrix(all_targets, all_preds)
    
    # Tăng kích thước ảnh lên 10x8 để chứa đủ 7 class mà không bị lộn xộn
    plt.figure(figsize=(10, 8))
    
    # Vẽ Heatmap
    sns.heatmap(
        cm, 
        annot=True,       # Hiển thị số lượng
        fmt='d',          # Định dạng số nguyên
        cmap='Blues',     # Dải màu xanh dương (giống ảnh trước của bạn)
        cbar=True,        # Hiện thanh tỷ lệ màu
        xticklabels=class_names, 
        yticklabels=class_names
    )
    
    # Căn chỉnh hiển thị
    plt.title('Confusion Matrix - FER2013 (7 Classes)', fontsize=16, pad=20)
    plt.xlabel('Dự đoán (Predicted)', fontsize=12, labelpad=10)
    plt.ylabel('Thực tế (True)', fontsize=12, labelpad=10)
    
    # Xoay nhãn trục X một góc 45 độ để chữ (như 'surprise', 'disgust') không bị đè lên nhau
    plt.xticks(rotation=45, ha='right') 
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    plt.show()

# ==============================================================================
# CÁCH GỌI HÀM SAU KHI TRAIN XONG (HOẶC KHI TEST)
# ==============================================================================

# 1. Đảm bảo load lại model có kết quả tốt nhất trước khi test
# model.load_state_dict(torch.load('cct_resnet_fer2013_best.pth'))

# 2. Gọi hàm đánh giá bằng tập Validation (val_loader)
evaluate_fer2013(model, val_loader, CFG['device'])